# 03 — Spectral VAE: encode/decode with topology

Train the spectral VAE on synthetic data. Compare reconstruction quality and spectral consistency.

The VAE has a dual-head encoder (spatial latent `z` + feature field `A`) and a topology-adaptive decoder conditioned on `c_spec`.

In [ ]:
import sys
sys.path.insert(0, "../src")

import torch
import matplotlib.pyplot as plt
from ald_sc.build_prior import build_arrow_prior
from ald_sc.vae import SpectralVAE
from ald_sc.losses import ALDSCLoss
from ald_sc.trainer import train_vae
from ald_sc.data import ToyImageDataset, build_dataloader

torch.manual_seed(3407)

## 1. Build prior and VAE

In [ ]:
F, q = 32, 8
embeddings = torch.randn(64, F)
prior = build_arrow_prior(embeddings, q=q, k=4)

vae = SpectralVAE(
    in_channels=3,
    latent_channels=4,
    feature_dim=F,
    base_channels=32,
)
loss_fn = ALDSCLoss(prior=prior, lambda_rec=1.0, lambda_chart=0.5, lambda_smooth=0.1)

print(f"Prior: F={prior.F}, q={prior.q}")
print(f"Trainable params: {sum(p.numel() for p in vae.parameters() if p.requires_grad):,}")

## 2. Train on synthetic data

In [ ]:
dataset = ToyImageDataset(num_samples=32, image_size=32, channels=3)
loader = build_dataloader(dataset, batch_size=8)

losses = list(train_vae(loader, vae, prior, loss_fn, epochs=20, lr=1e-3))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot([l["loss"] for l in losses], label="total")
ax.plot([l["rec"] for l in losses], label="rec", alpha=0.7)
ax.plot([l["chart"] for l in losses], label="chart", alpha=0.7)
ax.plot([l["smooth"] for l in losses], label="smooth", alpha=0.7)
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("VAE training losses")
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.savefig("../results/03_vae_losses.png", dpi=150)
plt.show()

## 3. Reconstruction visualization

In [ ]:
x = next(iter(loader))
with torch.no_grad():
    z, A, c_spec, x_hat = vae(x, prior)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    axes[0, i].imshow(x[i].permute(1, 2, 0).numpy() * 0.5 + 0.5)
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")
    axes[1, i].imshow(x_hat[i].permute(1, 2, 0).numpy() * 0.5 + 0.5)
    axes[1, i].set_title("Reconstruction")
    axes[1, i].axis("off")
plt.tight_layout()
plt.savefig("../results/03_reconstructions.png", dpi=150)
plt.show()

## 4. Spectral chart comparison

In [ ]:
with torch.no_grad():
    e_orig = prior.band_energies(A)
    _, A_hat, _, _ = vae(x, prior)
    e_recon = prior.band_energies(A_hat)

fig, ax = plt.subplots(figsize=(8, 3))
x_pos = torch.arange(q)
width = 0.35
ax.bar(x_pos - width/2, e_orig[0].numpy(), width, label="Original")
ax.bar(x_pos + width/2, e_recon[0].numpy(), width, label="Reconstructed")
ax.set_xlabel("Mode k")
ax.set_ylabel(r"$e_k$")
ax.set_title("Band energies: original vs reconstructed")
ax.legend()
plt.tight_layout()
plt.savefig("../results/03_band_energies.png", dpi=150)
plt.show()